# Minimum information tiers: paired omics

How many of the gene-metabolite associations of the MEANtools pilot (tomato) reach each tier, and what is missing in the graphs to reach the others?

**Case:** case 6, paired omics: transcript-metabolite associations (with cases 5 and 7). **Data:** the association graph (`shape_constraints/rdf/association.ttl`) and two **enrichment graphs of the pilot** (`pilots/pairedomics_overlap/results/`, built by `build_gene_annotation.py`): `gene_annotation.ttl` (for each gene its UniProt entry, its EC numbers and its orthologs among the enzymes of PlantMetWiki, from OrthoDB) and `feature_annotation.ttl` (the MS1 mass match of the features to PlantMetWiki metabolites). **Queries:** the files of [`sparql-examples-pmw`](https://github.com/pathway-lod/sparql-examples-pmw/tree/main/examples/MinimumInformationTiers) (<https://pathway-lod.github.io/sparql-examples-pmw/>), run in a local Oxigraph store, with PlantMetWiki, Rhea and UniProt called by `SERVICE` on the live endpoints.

The outputs are a **dated snapshot**: the date and the PlantMetWiki release are in the summary table at the end. Each query is run once with a time limit; a tier that did not finish is shown as *timeout*. The results of every tier are also saved as CSV in `results/`. To rerun, open the notebook with a kernel that has the packages of `requirements.txt` installed (see the README of this folder).

**What changed (2026-11-18).** The first version of this notebook found *zero* rows in almost every tier. The reason was **missing links in the graphs, not the biology**: the queries looked up the gene only by its Solyc locus (about 30 genes of PlantMetWiki), while the pilot also reaches pathways through the UniProt entry and the EC number of the gene (1,082 gene-pathway links, 311 genes), the gene-to-UniProt and ortholog links were not in the RDF, and the features of the MEANtools data had no annotation. The queries now have the three gene tiers of the pilot (locus, UniProt, EC) and the graphs have the links. Zero rows that remain are now a *computed* result and are explained under each tier. The pathway numbers (PC...) are those of the live PlantMetWiki release; match on the PlantCyc id when you compare with older snapshots (see notebook 06).

In [1]:
import mi
import pandas as pd
from IPython.display import display
pd.set_option('display.max_colwidth', 80)
CASE = '6 paired omics'
print('PlantMetWiki release:', mi.pmw_release())

PlantMetWiki release: r17.0.0_20260605-171052


## The data

In [2]:
data = ['association', 'gene_annotation', 'feature_annotation']
print('associations:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> SELECT (COUNT(?a) AS ?n) WHERE { ?a a lodp:GeneMetaboliteAssociation }'))
print('distinct genes:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> PREFIX sio: <http://semanticscience.org/resource/> SELECT (COUNT(DISTINCT ?g) AS ?n) WHERE { ?a a lodp:GeneMetaboliteAssociation ; sio:SIO_000628 ?g . FILTER(STRSTARTS(STR(?g), "https://plants.ensembl.org/")) }'))
print('functional clusters:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> PREFIX sio: <http://semanticscience.org/resource/> SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?a a lodp:GeneMetaboliteAssociation ; sio:SIO_000772/lodp:functionalCluster ?c }'))
Q = 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> SELECT (COUNT(DISTINCT ?g) AS ?n) WHERE { ?g %s }'
print('genes with a UniProt entry :', mi.local_count(data, Q % 'lodp:uniprot ?x'))
print('genes with an EC number    :', mi.local_count(data, Q % 'lodp:ecNumber ?x'))
print('genes with an ortholog among the enzymes of PlantMetWiki pathways (OrthoDB):', mi.local_count(data, Q % 'lodp:orthologOf ?x'))
print('features with an MS1 annotation:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> SELECT (COUNT(DISTINCT ?f) AS ?n) WHERE { ?a a lodp:MetaboliteAnnotation ; lodp:feature ?f }'))

associations: 1293


distinct genes: 350
functional clusters: 52


genes with a UniProt entry : 315


genes with an EC number    : 98
genes with an ortholog among the enzymes of PlantMetWiki pathways (OrthoDB): 23


features with an MS1 annotation: 43


## Gene and metabolite match the same pathway

`gene-and-metabolite-in-same-pathway`

gene-and-metabolite-in-same-pathway: An association of a gene and a metabolite feature where the gene (by its Solyc locus, its UniProt entry or, weakly, an EC number of its reactions, listed in PlantMetWiki) and the metabolite (by an InChIKey first-block match of an annotation of the feature) are in the same PlantMetWiki pathway. Both sides are computed here; the stored anchor terms are not used. NOTE: the metabolite side uses the MS1 mass match of `feature_annotation.ttl` (43 features); `gene-in-plantmetwiki-pathway` shows the gene side alone. OUTPUT: a list of genes (supported members of a pathway: a gene associated with a metabolite that is in the same pathway, matched by Solyc locus and InChIKey); the number of associations and of metabolite features that support each gene-pathway pair are metadata.

*Keywords:* InChIKey, MEANtools, Minimum Information: evidence tiers, PlantMetWiki pathway, Solyc locus, case 5: metabolomics, metabolite annotations, case 6: paired omics, transcript-metabolite associations, evidence tier, federated query, gene, gene-metabolite association, local evidence graph, metabolite annotation, output: genes, paired omics

In [3]:
r = mi.run('gene-and-metabolite-in-same-pathway', ['association', 'gene_annotation', 'feature_annotation'], CASE, timeout=400)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 16.7 s


,component,componentType,pathway,tier,supportingAssociations,supportingMetaboliteFeatures


**Reading.** No rows, a **computed** result that agrees with the pilot (`pairedomics_overlap`): **0 of 721** fully mapped pairs share a PlantMetWiki pathway, where about 5 would be expected by chance. Most fungus-induced features (oxylipins, acetylenic fatty acids) have no PlantCyc pathway, the metabolite identities rest on mass only (one mass matches several compounds, see notebook 06 for the isotope problem), and the gene side is mostly an EC number shared by many pathways. Notebook 06 follows one association that this tier misses and shows which other tier retrieves its pathway.

**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Gene side: the gene of an association is in a PlantMetWiki pathway

`gene-in-plantmetwiki-pathway`

gene-in-plantmetwiki-pathway: Gene side of `gene-and-metabolite-in-same-pathway`. The gene of each association is looked up in PlantMetWiki by its Solyc locus (tier gene_locus), by its UniProt entry (gene_uniprot) or by an EC number of its UniProt entry that is on a reaction of the pathway (gene_ec, weak), and the pathways are returned with the tier that found them. Counts the associations whose gene is in at least one pathway. OUTPUT: a list of genes (supported members of a pathway: the gene of an association is listed in a PlantMetWiki pathway by its Solyc locus), with the number of associations that support each gene-pathway pair as metadata.

*Keywords:* MEANtools, Minimum Information: evidence tiers, PlantMetWiki pathway, Solyc locus, case 6: paired omics, transcript-metabolite associations, evidence tier, federated query, gene, gene-metabolite association, local evidence graph, output: genes, paired omics

In [4]:
r = mi.run('gene-in-plantmetwiki-pathway', ['association', 'gene_annotation'], CASE, timeout=280)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

rows | 146 rows | 2.1 s


,component,componentType,pathway,tier,supportingAssociations
0,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc05g007260,gene,PC927,gene_ec,9
1,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc05g007260,gene,PC901,gene_ec,9
2,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc05g007260,gene,PC525,gene_ec,9
3,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc05g007260,gene,PC887,gene_ec,9
4,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc09g009190,gene,RC1004,gene_ec,7
5,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc11g006550,gene,PC951,gene_ec,7
6,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc08g066790,gene,PC910,gene_ec,7
7,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc08g066790,gene,PC483,gene_ec,7
8,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc01g102940,gene,PC349,gene_ec,7
9,https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc04g009620,gene,PC1029,gene_ec,7


**Reading.** Before, this tier found no gene (the locus alone reaches ~30 genes of PlantMetWiki, none of them in these associations). With the UniProt and the EC tiers of the pilot it finds a gene-pathway pair for the genes that have an EC number. Note the **tier column**: almost all hits come from the **EC number** (weak: an EC number is shared by many genes of unrelated pathways), a few from the UniProt entry or the locus. A gene that reaches a pathway only through an EC number is a lead, not a membership (weight 0.40 in the ranked query).

In [5]:
t = r['table']
t = t.assign(gene=t.component.str.extract(r'(Solyc\d+g\d+)')[0])
print('gene-pathway pairs:', len(t), '| genes:', t.gene.nunique(), '| pathways:', t.pathway.nunique())
print(t.groupby('tier').agg(pairs=('gene', 'size'), genes=('gene', 'nunique'), pathways=('pathway', 'nunique')))

gene-pathway pairs: 146 | genes: 47 | pathways: 126
         pairs  genes  pathways
tier                           
gene_ec    146     47       126


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Gene is an ortholog of a gene of a species that has the pathway

`ortholog-in-pathway-species`

ortholog-in-pathway-species: The metabolite matches a PlantMetWiki pathway (as in `gene-and-metabolite-in-same-pathway`) and the gene is an ortholog of a gene that PlantMetWiki lists in that pathway in another species. The orthology is read from the graph as lodp:orthologOf (for example from OrthoDB groups, see tier 5 of the pathway-component queries); the gene of the other species is matched to the pathway by its UniProt accession. Needs the gene annotation graph of the pilot (`gene_annotation.ttl`: lodp:orthologOf from OrthoDB groups that contain an enzyme listed in a PlantMetWiki pathway) and, for the MEANtools features, their MS1 annotation. On the MEANtools pilot it returns no rows: 23 of 315 genes have such an ortholog and 44 of their 88 associations involve an annotated feature, but the annotated metabolite is never in a pathway of the orthologous enzyme; `ranked-pathway-genes` shows the orthology alone. OUTPUT: a list of genes (putative members of a pathway: the gene is an ortholog of a gene listed in that pathway in another species, and its associated metabolite is in the pathway); the ortholog and the number of supporting associations are metadata.

*Keywords:* InChIKey, Minimum Information: evidence tiers, OrthoDB, PlantMetWiki pathway, UniProt, case 6: paired omics, transcript-metabolite associations, evidence tier, federated query, gene-metabolite association, local evidence graph, metabolite annotation, orthologs, output: genes, paired omics

In [6]:
r = mi.run('ortholog-in-pathway-species', ['association', 'gene_annotation', 'feature_annotation'], CASE, timeout=400)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 1.1 s


,component,componentType,pathway,orthologUniprot,supportingAssociations


**Reading.** No rows, a **computed** result. OrthoDB links 23 of the 315 genes to enzymes that PlantMetWiki lists in pathways, and 44 of the 88 associations of those genes involve an annotated feature, but the annotated metabolite is **never in a pathway of the orthologous enzyme**: it is the same finding as the pilot's 0 of 721. The orthology evidence alone (without the metabolite) is in `ranked-pathway-genes` (notebook 05).

**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Gene is predicted to catalyse the transformation between the two metabolites

`gene-catalyses-the-transformation`

gene-catalyses-the-transformation: A gene is associated with two metabolite features. Both are matched to PlantMetWiki metabolites (`annotation-matches-plantmetwiki`), and PlantMetWiki has a conversion from one to the other (substrate to product) catalysed by an enzyme that the gene is linked to (here by its UniProt accession, lodp:uniprot). Returns the gene, the two metabolites and the conversion. Needs the gene annotation graph (`gene_annotation.ttl`, lodp:uniprot) and annotated features (MS1 only for the MEANtools features). On the MEANtools pilot it returns no rows: 208 genes are linked to UniProt and associated with two annotated features, but PlantMetWiki has no conversion between two such metabolites catalysed by the enzyme of the gene. OUTPUT: a list of reactions (putative or supported: the PlantMetWiki conversion between two metabolites, catalysed by an enzyme that a co-expressed gene encodes) with their pathway; the gene, the enzyme (UniProt) and the two associated metabolite features (substrate and product) are metadata.

*Keywords:* InChIKey, Minimum Information: evidence tiers, PlantMetWiki reaction, UniProt, case 2: inferred enzyme function from PMN/PlantCyc+ db, case 5: metabolomics, metabolite annotations, case 6: paired omics, transcript-metabolite associations, enzyme, evidence tier, federated query, gene-metabolite association, local evidence graph, metabolite annotation, output: reactions, paired omics, substrate and product

In [7]:
r = mi.run('gene-catalyses-the-transformation', ['association', 'gene_annotation', 'feature_annotation'], CASE, timeout=400)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 36.1 s


,component,componentType,pathway,gene,substrateFeature,productFeature,uniprot


**Reading.** No rows, and this is a **result**: 208 genes have a UniProt link and are associated with two annotated features, but PlantMetWiki has **no conversion between the two matched metabolites that is catalysed by the enzyme of that gene**. The annotation is MS1 only (one mass matches several compounds, so most pairs are not real substrate-product pairs) and the gene-to-enzyme link is the UniProt entry of the gene, not a measured activity. This tier is meant for **annotated** (MS/MS) data, such as the Brassicaceae pilot, once it has paired transcriptomics.

**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Ortholog in a pathway species and a machine-learning predicted transformation

`ortholog-plus-ml-prediction`

ortholog-plus-ml-prediction: As in `ortholog-in-pathway-species`, and the transformation also matches an enzymatic transformation predicted by a model on the PlantMetWiki graph (case 7, for example the predictions of PlantMetBench): a predicted link between the enzyme the gene is an ortholog of and a reaction that connects the two metabolites. The prediction is read as lodp:PredictedLink with lodp:predictionScore and the model run. NOT RUNNABLE ON THE CURRENT GRAPHS: no predictions are in the RDF yet. OUTPUT: a list of reactions (putative: a reaction predicted by a machine-learning model for an enzyme that the gene is an ortholog of), with the pathway when it is known; the gene, the enzyme, the model run and the prediction score are metadata.

*Keywords:* Minimum Information: evidence tiers, case 6: paired omics, transcript-metabolite associations, case 7: unsupervised GNN predictions of reactions (conversion steps), evidence tier, gene-metabolite association, local evidence graph, machine learning prediction, no data yet, orthologs, output: reactions, paired omics, predicted enzymatic transformation

In [8]:
r = mi.run('ortholog-plus-ml-prediction', ['association', 'gene_annotation', 'feature_annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 0.0 s


,component,componentType,pathway,gene,enzyme,score,run


**Reading.** Still **no data**: there are no predictions of the machine-learning models (PlantMetBench) in the graphs (`unsupervised_overlap` is a plan). The tier is encoded and will run when the predictions are added.

**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Summary of this case

In [9]:
s = pd.read_csv('results/summary.csv')
display(s[s['case'] == CASE].reset_index(drop=True))

,tier,case,status,n_rows,seconds,timeout_s,date,pmw_release
0,gene-and-metabolite-in-same-pathway,6 paired omics,no rows,0.0,16.7,400,2026-11-18,r17.0.0_20260605-171052
1,gene-catalyses-the-transformation,6 paired omics,no rows,0.0,36.1,400,2026-11-18,r17.0.0_20260605-171052
2,gene-in-plantmetwiki-pathway,6 paired omics,rows,146.0,2.1,280,2026-11-18,r17.0.0_20260605-171052
3,ortholog-in-pathway-species,6 paired omics,no rows,0.0,1.1,400,2026-11-18,r17.0.0_20260605-171052
4,ortholog-plus-ml-prediction,6 paired omics,no rows,0.0,0.0,150,2026-11-18,r17.0.0_20260605-171052
